# AIS 피처 인코딩 확인

현재 5-dim 인코딩과 개선된 8-dim 인코딩을 비교합니다.

- 센티넬 값 처리 (SOG==0, COG==360, HDG==0 or >360)
- sin/cos 마스킹
- sog_zero / cog_missing / hdg_missing 플래그

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = '../data' if os.path.isdir('../data/train') else 'data'
df = pd.read_csv(BASE + '/train/train.csv')

sog = df['sog'].astype(float).values
cog = df['cog'].astype(float).values
hdg = df['true_heading'].astype(float).values

print('샘플 수:', len(df))
print('컬럼:', list(df.columns))

## 1. 센티넬 값 현황

In [ ]:
print('=== 센티넬 / 결측 현황 ===')
print(f'SOG == 0    : {(sog==0).sum():6d}  ({(sog==0).mean()*100:.2f}%)')
print(f'COG == 360  : {(cog==360).sum():6d}  ({(cog==360).mean()*100:.2f}%)')
print(f'HDG == 0    : {(hdg==0).sum():6d}  ({(hdg==0).mean()*100:.2f}%)')
print(f'HDG > 360   : {(hdg>360).sum():6d}  ({(hdg>360).mean()*100:.2f}%)')
print()
print('→ 이 값들은 실제 측정값이 아닌 "모름" 표시 (센티넬)')

## 2. 기존 인코딩 (5-dim) vs 개선 인코딩 (8-dim)

In [ ]:
# ── 기존 5-dim ────────────────────────────────────────────────────────────────
SOG_MEAN_OLD, SOG_STD_OLD = 10.0, 8.0

def encode_old(sog, cog, hdg):
    sog_norm = (sog - SOG_MEAN_OLD) / (SOG_STD_OLD + 1e-6)
    cog_rad  = np.deg2rad(cog)
    hdg_rad  = np.deg2rad(hdg)
    return np.stack([
        sog_norm,
        np.sin(cog_rad), np.cos(cog_rad),
        np.sin(hdg_rad), np.cos(hdg_rad),
    ], axis=1)

# ── 개선 8-dim ────────────────────────────────────────────────────────────────
# SOG: log-clip 통계는 train 전체 기준
sog_log = np.log1p(np.clip(sog, 0, 30))
SOG_MEAN_NEW = float(sog_log.mean())
SOG_STD_NEW  = float(sog_log.std() + 1e-6)
print(f'SOG log-clip 통계: mean={SOG_MEAN_NEW:.4f}, std={SOG_STD_NEW:.4f}')

def encode_new(sog, cog, hdg):
    sog_norm    = (np.log1p(np.clip(sog, 0, 30)) - SOG_MEAN_NEW) / SOG_STD_NEW
    sog_zero    = (sog == 0.0).astype(np.float32)
    cog_missing = ((cog == 360.0) | (sog == 0.0)).astype(np.float32)
    hdg_missing = ((hdg == 0.0) | (hdg > 360.0)).astype(np.float32)

    cog_rad = np.deg2rad(cog)
    hdg_rad = np.deg2rad(hdg)
    cog_sin = np.where(cog_missing, 0.0, np.sin(cog_rad))
    cog_cos = np.where(cog_missing, 0.0, np.cos(cog_rad))
    hdg_sin = np.where(hdg_missing, 0.0, np.sin(hdg_rad))
    hdg_cos = np.where(hdg_missing, 0.0, np.cos(hdg_rad))

    return np.stack([
        sog_norm, cog_sin, cog_cos, hdg_sin, hdg_cos,
        sog_zero, cog_missing, hdg_missing,
    ], axis=1)

old = encode_old(sog, cog, hdg)
new = encode_new(sog, cog, hdg)

print(f'\n기존 shape: {old.shape}  →  개선 shape: {new.shape}')
print('\n피처 이름 (8-dim):')
FEAT_NAMES = ['sog_norm','cog_sin','cog_cos','hdg_sin','hdg_cos',
              'sog_zero','cog_missing','hdg_missing']
for i, n in enumerate(FEAT_NAMES):
    print(f'  [{i}] {n}')

## 3. 샘플 비교 — 정지 배 vs 이동 배

In [ ]:
# 정지 배 (SOG==0) 샘플
stopped_idx = np.where(sog == 0)[0][0]
# 이동 중 배 (SOG>0, HDG 유효) 샘플
moving_idx  = np.where((sog > 2) & (hdg > 0) & (hdg <= 360) & (cog != 360))[0][0]

print('=== 정지 배 (SOG=0) ===')
print(f'  SOG={sog[stopped_idx]:.1f}, COG={cog[stopped_idx]:.1f}, HDG={hdg[stopped_idx]:.1f}')
print('  기존 5-dim:', np.round(old[stopped_idx], 4))
print('  개선 8-dim:', np.round(new[stopped_idx], 4))
print('  → sog_zero=1, cog_missing=1 (COG 마스킹됨)')

print()
print('=== 이동 중 배 ===')
print(f'  SOG={sog[moving_idx]:.1f}, COG={cog[moving_idx]:.1f}, HDG={hdg[moving_idx]:.1f}')
print('  기존 5-dim:', np.round(old[moving_idx], 4))
print('  개선 8-dim:', np.round(new[moving_idx], 4))
print('  → 모든 플래그 0 (유효 측정값)')

## 4. 피처 분포 시각화

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, (name, ax) in enumerate(zip(FEAT_NAMES, axes)):
    vals = new[:, i]
    if name in ('sog_zero', 'cog_missing', 'hdg_missing'):
        # 플래그: 0/1 bar chart
        counts = [int((vals==0).sum()), int((vals==1).sum())]
        ax.bar(['0 (유효)', '1 (무효/정지)'], counts, color=['#4C72B0','#DD8452'])
        ax.set_title(name)
        for j, c in enumerate(counts):
            ax.text(j, c, f'{c/len(vals)*100:.1f}%', ha='center', va='bottom', fontsize=9)
    else:
        ax.hist(vals, bins=60, color='#4C72B0', alpha=0.8)
        ax.set_title(f'{name}\nmean={vals.mean():.3f}, std={vals.std():.3f}')
    ax.set_xlabel('value')

plt.suptitle('개선 8-dim AIS 인코딩 분포', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. 기존 vs 개선 — HDG 인코딩 차이

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 기존: HDG==0인 샘플의 hdg_sin 분포 (센티넬인데 sin(0)=0으로 들어감)
null_hdg = hdg == 0
axes[0].hist(old[null_hdg, 3], bins=30, color='#DD8452', alpha=0.8)
axes[0].set_title(f'기존: HDG==0 샘플의 hdg_sin\n({null_hdg.sum()}개, 센티넬이지만 sin(0)=0으로 처리됨)')
axes[0].set_xlabel('hdg_sin 값')

# 개선: HDG==0인 샘플은 hdg_sin=0, hdg_missing=1
axes[1].bar(['hdg_sin=0\n(마스킹)', 'hdg_missing=1\n(플래그)'],
            [(new[null_hdg, 3]==0).sum(), (new[null_hdg, 7]==1).sum()],
            color=['#4C72B0','#55A868'])
axes[1].set_title(f'개선: HDG==0 샘플 처리\n({null_hdg.sum()}개 모두 명시적으로 "모름" 표시)')

plt.tight_layout()
plt.show()

print('\n=== 요약 ===')
print(f'HDG==0 샘플: {null_hdg.sum()}개 ({null_hdg.mean()*100:.1f}%)')
print('기존: sin(0°)=0.0, cos(0°)=1.0 으로 "북쪽"처럼 학습됨')
print('개선: hdg_sin=0, hdg_cos=0, hdg_missing=1 으로 "모름"명시')